
# SBA Loan Default Scoring — Project 2 (Scoring Notebook)

**Purpose:**  
This notebook generates default predictions for new data using the trained XGBoost model and saved artifacts, strictly following the Project 2 scoring contract.  
- **Inputs:** DataFrame with same schema as training data (minus `MIS_Status`).
- **Outputs:** DataFrame with columns: `index`, `label`, `probability_0`, `probability_1`.
- All cleaning, feature engineering, and encodings are identical to the training pipeline; no fitting or model retraining occurs here.

In [ ]:
import json, pickle, numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path


In [ ]:
ART_DIR   = Path("./artifacts")  # model + encoders + features + meta
DATA_PATH = Path("..") / "data" / "raw" / "SBA_loans_project_2_holdout_students_valid.csv"
OUT_DIR   = Path("../outputs")
OUT_P2     = OUT_DIR / "p2_scoring_output.csv"
OUT_KAGGLE = OUT_DIR / "submission_final.csv"

In [ ]:
with open(ART_DIR / "params_rounds_threshold.json") as f:
    meta = json.load(f)
macro_f1_threshold = float(meta["macro_f1_threshold"])  # use this for label
print("Loaded macro-F1 threshold:", macro_f1_threshold)


Loaded macro-F1 threshold: 0.5199999999999999


In [ ]:

woe_encoder = pickle.load(open(ART_DIR / "woe_encoder.pkl", "rb"))
te_encoder  = pickle.load(open(ART_DIR / "target_encoder.pkl", "rb"))
booster = xgb.Booster()
booster.load_model(str(ART_DIR / "model_xgb.json"))  
print("Artifacts loaded: encoders + model_xgb.json")


Artifacts loaded: encoders + model_xgb.json


In [ ]:
def pre_split_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    money_keys = ["grappv","sba_appv","disbursement","gross","chgoff","balance","prin","amount","appv"]
    money_cols = [c for c in df.columns if any(k in c.lower() for k in money_keys)]
    for c in money_cols:
        df[c] = (df[c].astype(str)
                        .str.replace(r"[\$,]", "", regex=True)
                        .replace({"": np.nan, "nan": np.nan, "NONE": np.nan, "None": np.nan}))
        df[c] = pd.to_numeric(df[c], errors="coerce")


    for c in ["City","State","Bank","BankState","RevLineCr","LowDoc"]:
        if c in df.columns:
            df[c] = (df[c].astype(str).str.strip().str.upper()
                             .replace({"": np.nan, "NAN": np.nan, "NONE": np.nan}))

    if "NewExist" in df.columns:
        df["NewExist"] = pd.to_numeric(df["NewExist"], errors="coerce")

    return df

holdout = pd.read_csv(DATA_PATH)
holdout = pre_split_clean(holdout)
print("Holdout shape after hygiene:", holdout.shape)

Holdout shape after hygiene: (99808, 19)


In [ ]:
def _sd(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce").replace(0, np.nan)
    return (a / b).replace([np.inf, -np.inf], np.nan).fillna(0)

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    m = df.copy()

    m['SBA_to_GrAppv']       = _sd(m.get('SBA_Appv'), m.get('GrAppv'))
    m['Disburse_to_Approve'] = _sd(m.get('DisbursementGross'), m.get('GrAppv'))
    m['RetainJob_Rate']      = _sd(m.get('RetainedJob'), m.get('NoEmp'))
    m['GrAppv_per_emp']      = _sd(m.get('GrAppv'), m.get('NoEmp'))
    m['Disb_per_emp']        = _sd(m.get('DisbursementGross'), m.get('NoEmp'))
    m['Undisbursed_ratio']   = _sd(m.get('GrAppv') - m.get('DisbursementGross'), m.get('GrAppv')).clip(lower=0)
    m['SBA_shortfall_ratio'] = _sd(m.get('GrAppv') - m.get('SBA_Appv'), m.get('GrAppv')).clip(lower=0)
    m['Chargeoff_ratio']     = _sd(m.get('ChgOffPrinGr'), m.get('GrAppv'))
    m['Balance_to_disb']     = _sd(m.get('BalanceGross'), m.get('DisbursementGross'))


    m['Gross_gap']  = (pd.to_numeric(m.get('GrAppv'), errors="coerce") -
                       pd.to_numeric(m.get('DisbursementGross'), errors="coerce")).fillna(0)
    m['SBA_gap']    = (pd.to_numeric(m.get('GrAppv'), errors="coerce") -
                       pd.to_numeric(m.get('SBA_Appv'), errors="coerce")).fillna(0)
    m['Jobs_total'] = (pd.to_numeric(m.get('CreateJob'), errors="coerce").fillna(0) +
                       pd.to_numeric(m.get('RetainedJob'), errors="coerce").fillna(0))
    m['Jobs_per_100k_disb'] = _sd(m['Jobs_total'],
                                  (pd.to_numeric(m.get('DisbursementGross'), errors="coerce")/100000.0)+1)

    m['Util_x_SBA'] = pd.to_numeric(m['Disburse_to_Approve'], errors="coerce").fillna(0) * \
                      pd.to_numeric(m['SBA_to_GrAppv'], errors="coerce").fillna(0)
    return m

Xh = add_engineered_features(holdout)
print("Holdout with engineered features:", Xh.shape)

Holdout with engineered features: (99808, 33)


In [ ]:
woe_features_all = ['State','BankState','NewExist','UrbanRural','LowDoc']
te_features_all  = ['City','Zip','Bank','NAICS','FranchiseCode']

woe_features = [c for c in woe_features_all if c in Xh.columns]
te_features  = [c for c in te_features_all  if c in Xh.columns]

Xh_woe = woe_encoder.transform(Xh[woe_features])
Xh_woe.columns = [f"{c}_woe" for c in woe_features]

Xh_te  = te_encoder.transform(Xh[te_features])
Xh_te.columns  = [f"{c}_trg" for c in te_features]

drop_cols = list(set(woe_features + te_features + ['RevLineCr']) & set(Xh.columns))
X_holdout_final = pd.concat([
    Xh.drop(columns=drop_cols),
    Xh_woe, Xh_te
], axis=1)

print("X_holdout_final before align:", X_holdout_final.shape)


X_holdout_final before align: (99808, 32)


In [ ]:
feature_list = [ln.strip() for ln in open(ART_DIR / "features.txt")]
for c in feature_list:
    if c not in X_holdout_final.columns:
        X_holdout_final[c] = 0.0
X_holdout_final = X_holdout_final[feature_list].astype('float32')

print("X_holdout_final aligned:", X_holdout_final.shape)


X_holdout_final aligned: (99808, 28)


In [ ]:
dhold = xgb.DMatrix(X_holdout_final)
p1 = booster.predict(dhold).astype(np.float64)
p0 = 1.0 - p1

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_P2     = OUT_DIR / "p2_scoring_output_final.csv"
OUT_KAGGLE = OUT_DIR / "submission_final_v1.csv"

record_id = holdout["index"] if "index" in holdout.columns else holdout.index

assert list(scored.columns) == ["index", "label", "probability_0", "probability_1"], "Column order mismatch!"
assert scored.shape[0] == holdout.shape[0], "Row count mismatch!"


scored = pd.DataFrame({
    "index": record_id,
    "label": (p1 >= macro_f1_threshold).astype(np.int16),
    "probability_0": p0,
    "probability_1": p1
})
scored.to_csv(OUT_P2, index=False)

id_col = "LoanNr_ChkDgt" if "LoanNr_ChkDgt" in holdout.columns else None
kaggle_id = holdout[id_col].values if id_col else record_id

kaggle = pd.DataFrame({
    "ID": kaggle_id,
    "probability_1": p1
})
kaggle.to_csv(OUT_KAGGLE, index=False)

print(f"Wrote: {OUT_P2} ({len(scored)} rows) and {OUT_KAGGLE} ({len(kaggle)} rows)")


Wrote: ..\outputs\p2_scoring_output_final.csv (99808 rows) and ..\outputs\submission_final_v1.csv (99808 rows)
